<a href="https://colab.research.google.com/github/awaisfarooqchaudhry/IB9AU-GenerativeAI-2026/blob/main/Task17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 17 — Local smolagents Agent for Sharpe Ratio Comparison (Google Colab)

This notebook is a **complete beginner-friendly Colab notebook** for **Required Task 17**.

It is adapted from your uploaded **`Agents3.ipynb`**, but cleaned up and optimized for this specific assignment task.

## What Task 17 asks you to do
You need to **write a prompt** for a **local smolagents (Qwen 3B) agent** so it can autonomously:

1. download daily closing prices for the last **180 days**
2. for a **Tech portfolio**:
   - NVDA
   - AAPL
   - MSFT
3. for a **Bank portfolio**:
   - JPM
   - BAC
   - C
4. calculate:
   - daily returns
   - Sharpe ratio for each stock using  
     **Mean Daily Return / Std Dev of Daily Returns × sqrt(252)**
5. create a bar chart comparing all 6 Sharpe ratios
6. color the bars:
   - **Green** for Tech
   - **Blue** for Banks
7. save the chart as:
   - **`sharpe_comparison.png`**

## What this notebook does
- installs `smolagents` and the local Transformers backend
- loads **Qwen 2.5 Coder 3B Instruct** locally on Colab GPU
- initializes a **CodeAgent**
- defines the final prompt for Task 17
- runs the prompt through the agent
- displays the saved chart
- runs a **manual verification / fallback** calculation in Python
- saves:
  - `sharpe_comparison.png`
  - `task17_sharpe_table.csv`
  - `task17_summary.txt`

## Colab recommendation
Use **Runtime → Change runtime type → T4 GPU**

## Important
Run the notebook **from top to bottom**.
Do not skip cells.


## Why this notebook follows the lab structure

Your uploaded `Agents3.ipynb` uses:
- `smolagents`
- `TransformersModel`
- `CodeAgent`
- a local **Qwen 3B** coding model

This notebook keeps that exact pattern, but narrows it to the Sharpe-ratio assignment and adds a safer verification section.


## Step 0 — Install packages

This is a Colab-safe install cell.
It avoids upgrading core packages like pandas and numpy unnecessarily.


In [ ]:
# Colab-safe install cell with quantization support
!pip -q install "smolagents[transformers]" yfinance matplotlib bitsandbytes accelerate

print("✅ Packages installed.")
print("This notebook uses 4-bit quantization to fit the local Qwen 3B model on a T4 more reliably.")

## Step 1 — Check GPU


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected. The notebook may still work, but local model inference will be much slower.")

## Step 2 — Optional Hugging Face token

The Qwen model is public, so a token is often not strictly required.
But if download limits or Hub access cause issues, you can add your Hugging Face token here.


In [ ]:
import os
from getpass import getpass

USE_HF_TOKEN = False

if USE_HF_TOKEN:
    os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face token: ")
    print("✅ HF token set for this session.")
else:
    print("Skipping HF token setup.")

## Step 3 — Imports


In [ ]:
import os
import math
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import torch
from transformers import BitsAndBytesConfig

from smolagents import CodeAgent, TransformersModel

## Step 4 — Load the local Qwen 3B model

This follows the same local-agent pattern as the lab notebook, using a small coding model that can run on a Colab T4 GPU.


### Why this fixes the OOM

The original notebook loaded the 3B model in regular FP16, which can be too tight on a T4 once the agent starts generating over a long prompt and multi-step memory.

This patched version:
- uses **4-bit bitsandbytes quantization**
- lowers `max_new_tokens`
- lowers `max_steps`
- enables PyTorch expandable CUDA segments

That combination is much more reliable on Colab T4 GPUs.

In [ ]:
print("⬇️ Loading Qwen 2.5 Coder 3B in 4-bit mode...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = TransformersModel(
    model_id="Qwen/Qwen2.5-Coder-3B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
    max_new_tokens=768,
    temperature=0.1,
    model_kwargs={
        "quantization_config": bnb_config,
        "low_cpu_mem_usage": True
    }
)

print("✅ Local quantized model loaded.")

## Step 5 — Initialize the CodeAgent

We authorize only the imports needed for this assignment:
- `yfinance`
- `pandas`
- `numpy`
- `matplotlib.pyplot`
- `os`
- `math`


In [ ]:
agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=4,
    additional_authorized_imports=[
        "yfinance",
        "pandas",
        "numpy",
        "matplotlib.pyplot",
        "os",
        "math"
    ]
)

print("✅ CodeAgent initialized.")

## Step 6 — The final Task 17 prompt

This is the exact prompt you can show as your assignment answer.
It tells the agent exactly what to do, what formula to use, how to color the chart, and what file to save.


In [ ]:
task17_prompt = """
You are a quantitative analyst working autonomously in Python.

Your task is to compare Big Tech vs Big Banks on a risk-adjusted basis using daily closing prices from the last 180 calendar days.

Requirements:
1. Download daily closing prices using yfinance for these two groups:

Tech portfolio:
- NVDA
- AAPL
- MSFT

Bank portfolio:
- JPM
- BAC
- C

2. Build a DataFrame of daily closing prices.

3. Calculate daily returns for each stock using percentage change.

4. Calculate the Sharpe Ratio for each stock using this exact formula:
Sharpe Ratio = Mean Daily Return / Standard Deviation of Daily Returns * sqrt(252)

Assume the risk-free rate is 0.

5. Create a bar chart comparing the Sharpe Ratios of all 6 stocks.

6. Color the bars:
- Green for the tech stocks
- Blue for the bank stocks

7. Label the axes clearly and add a title.

8. Save the figure as:
sharpe_comparison.png

9. Print a neat table of Sharpe Ratios.

10. Compute the average Sharpe Ratio for the Tech group and for the Bank group, and state clearly which sector performed better on this simple risk-adjusted basis.

Return a concise final summary after completing the analysis.
""".strip()

print(task17_prompt)

## Step 7 — Run the agent on the Task 17 prompt

This is the main assignment execution step.


In [ ]:
print("🤖 Agent is working...")

agent_result = None
agent_error = None

try:
    agent_result = agent.run(task17_prompt, stream=False)
    print("\n===== AGENT FINAL RESPONSE =====")
    print(agent_result)
except Exception as e:
    agent_error = str(e)
    print("\n⚠️ Agent run failed.")
    print(agent_error)
    print("\nThe notebook will continue to the manual verification section, which still completes the assignment outputs.")

## Step 8 — Display the chart generated by the agent

If the agent saved `sharpe_comparison.png` successfully, this cell will display it.


In [ ]:
from IPython.display import Image, display

if os.path.exists("sharpe_comparison.png"):
    print("✅ Found sharpe_comparison.png")
    display(Image("sharpe_comparison.png"))
else:
    print("⚠️ sharpe_comparison.png does not exist yet.")
    print("That is okay — the manual verification section below will create it if needed.")

## Step 9 — Manual verification and fallback calculation

This section computes the same Sharpe ratios directly in Python.

Why include this?
- It verifies the agent's result
- It gives you a transparent audit trail
- If the agent fails to save the chart, this section will create it anyway


In [ ]:
TECH_TICKERS = ["NVDA", "AAPL", "MSFT"]
BANK_TICKERS = ["JPM", "BAC", "C"]
ALL_TICKERS = TECH_TICKERS + BANK_TICKERS

price_df = yf.download(
    tickers=ALL_TICKERS,
    period="180d",
    interval="1d",
    auto_adjust=False,
    progress=False
)

# yfinance can return a multi-index column structure
if isinstance(price_df.columns, pd.MultiIndex):
    if "Close" in price_df.columns.get_level_values(0):
        close_df = price_df["Close"].copy()
    else:
        # fallback: use Adj Close if Close not present
        close_df = price_df["Adj Close"].copy()
else:
    # fallback for single-ticker style edge cases
    close_df = price_df[["Close"]].copy()
    close_df.columns = ALL_TICKERS[:1]

close_df = close_df.dropna(how="all")

returns_df = close_df.pct_change().dropna(how="all")

sharpe_ratios = returns_df.mean() / returns_df.std()
sharpe_ratios = sharpe_ratios * np.sqrt(252)

sharpe_table = pd.DataFrame({
    "ticker": sharpe_ratios.index,
    "sharpe_ratio": sharpe_ratios.values
})

sharpe_table["sector"] = sharpe_table["ticker"].apply(
    lambda x: "Tech" if x in TECH_TICKERS else "Bank"
)

sharpe_table = sharpe_table.sort_values(by="sharpe_ratio", ascending=False).reset_index(drop=True)

tech_avg_sharpe = sharpe_table.loc[sharpe_table["sector"] == "Tech", "sharpe_ratio"].mean()
bank_avg_sharpe = sharpe_table.loc[sharpe_table["sector"] == "Bank", "sharpe_ratio"].mean()

display(sharpe_table)

print("Average Tech Sharpe :", round(float(tech_avg_sharpe), 4))
print("Average Bank Sharpe :", round(float(bank_avg_sharpe), 4))

if tech_avg_sharpe > bank_avg_sharpe:
    print("✅ Manual check: Big Tech performed better on this simple risk-adjusted basis.")
elif bank_avg_sharpe > tech_avg_sharpe:
    print("✅ Manual check: Big Banks performed better on this simple risk-adjusted basis.")
else:
    print("✅ Manual check: Both sectors are tied on this simple risk-adjusted basis.")

## Step 10 — Create / overwrite the required chart if needed

This cell ensures the required file **`sharpe_comparison.png`** exists, even if the agent did not save it properly.


In [ ]:
bar_colors = ["green" if t in TECH_TICKERS else "blue" for t in sharpe_table["ticker"]]

plt.figure(figsize=(10, 6))
plt.bar(sharpe_table["ticker"], sharpe_table["sharpe_ratio"], color=bar_colors)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Ticker")
plt.ylabel("Sharpe Ratio")
plt.title("Sharpe Ratio Comparison: Big Tech vs Big Banks (Last 180 Days)")
plt.tight_layout()
plt.savefig("sharpe_comparison.png", dpi=200)
plt.show()

print("✅ Saved sharpe_comparison.png")

## Step 11 — Save results to files

This creates:
- `task17_sharpe_table.csv`
- `task17_summary.txt`
- `task17_outputs/` folder


In [ ]:
output_dir = Path("task17_outputs")
output_dir.mkdir(exist_ok=True)

sharpe_table.to_csv(output_dir / "task17_sharpe_table.csv", index=False)

if tech_avg_sharpe > bank_avg_sharpe:
    sector_winner = "Big Tech"
elif bank_avg_sharpe > tech_avg_sharpe:
    sector_winner = "Big Banks"
else:
    sector_winner = "Tie"

summary_text = f"""
Task 17 Summary

Assignment question:
Which sector has performed better this year on a risk-adjusted basis: Big Tech or Big Banks?

Tickers used:
- Tech: NVDA, AAPL, MSFT
- Banks: JPM, BAC, C

Method:
- Download daily closing prices for the last 180 days
- Compute daily returns
- Compute Sharpe Ratio = Mean Daily Return / Std Dev of Daily Returns * sqrt(252)
- Compare individual stocks and average sector Sharpe

Average Tech Sharpe: {float(tech_avg_sharpe):.6f}
Average Bank Sharpe: {float(bank_avg_sharpe):.6f}

Conclusion:
{sector_winner} performed better on this simple Sharpe-ratio basis over the last 180 days.

Files created:
- sharpe_comparison.png
- task17_outputs/task17_sharpe_table.csv
- task17_outputs/task17_summary.txt
""".strip()

with open(output_dir / "task17_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)

## Step 12 — Download the outputs


In [ ]:
from google.colab import files

files.download("sharpe_comparison.png")
files.download("task17_outputs/task17_sharpe_table.csv")
files.download("task17_outputs/task17_summary.txt")

## Step 13 — Optional: zip the outputs folder


In [ ]:
import shutil

archive_path = shutil.make_archive("task17_outputs_archive", "zip", "task17_outputs")
print("✅ Created archive:", archive_path)

from google.colab import files
files.download(archive_path)

## Troubleshooting

### If the model download is slow
That is normal on the first run.

### If the agent fails to save the chart
Use the manual verification section.
It will still create `sharpe_comparison.png`.

### If Colab runs out of memory
Restart the runtime and try again.
A T4 GPU is usually enough for this 3B model.

### If yfinance returns missing prices
Re-run the cell once.
Yahoo Finance sometimes has temporary response issues.

### If you only need the assignment prompt
You can stop after **Step 6**, because that cell contains the prompt itself.
